In [ ]:

import sys
import subprocess
import json
import textwrap
from pathlib import Path
from datetime import datetime
from xml.sax.saxutils import escape

try:
    import pandas as pd
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "reportlab", "pyarrow"])
    import pandas as pd
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )

# ============================================================
# 1) PROJECT ROOT DISCOVERY
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
REPORTS_DIR = PROJECT_ROOT / "reports"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_FILE_NAME = "02 Data Collection and Ingestion- DM4ML-Group51.pdf"
OUTPUT_PATH = PROJECT_ROOT / OUTPUT_FILE_NAME

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_ROOT: {RAW_ROOT}")
print(f"BRONZE_ROOT: {BRONZE_ROOT}")

# ============================================================
# 2) HELPERS
# ============================================================
def safe_str(x):
    try:
        return str(x)
    except Exception:
        return ""

def rel_path(path):
    try:
        return safe_str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return safe_str(path)

def latest_file(base_dir, patterns):
    if not base_dir.exists():
        return None
    matches = []
    for pattern in patterns:
        matches.extend(base_dir.rglob(pattern))
    if not matches:
        return None
    return max(matches, key=lambda p: p.stat().st_mtime)

def collect_files(base_dir, patterns):
    files = []
    if base_dir.exists():
        for pattern in patterns:
            files.extend(base_dir.rglob(pattern))
    return sorted(set(files))

def file_info(path):
    if path is None or not path.exists():
        return None
    stat = path.stat()
    return {
        "name": path.name,
        "path": safe_str(path),
        "size_kb": round(stat.st_size / 1024, 2),
        "modified": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
    }

def summarize_csv(path, n=5):
    try:
        df = pd.read_csv(path, nrows=n)
        return {
            "columns": list(df.columns),
            "sample_rows": len(df),
            "error": None,
        }
    except Exception as e:
        return {
            "columns": [],
            "sample_rows": 0,
            "error": safe_str(e),
        }

def summarize_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        if isinstance(payload, dict):
            top_keys = list(payload.keys())[:20]
            sample_keys = []
            for _, v in payload.items():
                if isinstance(v, list) and v and isinstance(v[0], dict):
                    sample_keys = list(v[0].keys())
                    break
            return {
                "type": "dict",
                "top_keys": top_keys,
                "sample_keys": sample_keys,
                "error": None,
            }

        if isinstance(payload, list):
            sample_keys = list(payload[0].keys()) if payload and isinstance(payload[0], dict) else []
            return {
                "type": "list",
                "top_keys": [],
                "sample_keys": sample_keys,
                "list_length": len(payload),
                "error": None,
            }

        return {
            "type": type(payload).__name__,
            "top_keys": [],
            "sample_keys": [],
            "error": None,
        }

    except Exception as e:
        return {
            "type": "unknown",
            "top_keys": [],
            "sample_keys": [],
            "error": safe_str(e),
        }

def summarize_parquet(path):
    try:
        df = pd.read_parquet(path)
        return {
            "rows": len(df),
            "columns": list(df.columns),
            "error": None,
        }
    except Exception as e:
        return {
            "rows": None,
            "columns": [],
            "error": safe_str(e),
        }

def find_log_files():
    patterns = [
        "*.log",
        "*.txt",
        "*report*.json",
        "*summary*.csv",
        "*issues*.csv",
        "*fix*.csv",
        "*report*.pdf",
    ]
    matches = []
    if REPORTS_DIR.exists():
        for pattern in patterns:
            matches.extend(REPORTS_DIR.rglob(pattern))
    if SRC_DIR.exists():
        matches.extend(SRC_DIR.rglob("run_validation.txt"))
    return sorted(set(matches))

def read_text_preview(path, max_lines=30):
    if not path or not path.exists():
        return "File not found."
    try:
        lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
        return "\n".join(lines[:max_lines])
    except Exception as e:
        return f"Could not preview file: {e}"

def build_tree_text(base_path, max_depth=5, max_items=200):
    if not base_path.exists():
        return f"{base_path.name}/ (not found)"
    lines = [f"{base_path.name}/"]
    count = 0

    def walk(path, prefix="", depth=0):
        nonlocal count
        if depth >= max_depth or count >= max_items:
            return
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for idx, item in enumerate(items):
            if count >= max_items:
                break
            connector = "└── " if idx == len(items) - 1 else "├── "
            lines.append(prefix + connector + item.name + ("/" if item.is_dir() else ""))
            count += 1
            if item.is_dir():
                extension = "    " if idx == len(items) - 1 else "│   "
                walk(item, prefix + extension, depth + 1)

    walk(base_path)
    if count >= max_items:
        lines.append("... output truncated ...")
    return "\n".join(lines)

def wrap_block_text(text, width=95):
    lines = []
    for line in str(text).splitlines():
        if not line.strip():
            lines.append("")
            continue
        wrapped = textwrap.wrap(
            line,
            width=width,
            break_long_words=True,
            break_on_hyphens=True,
            replace_whitespace=False,
            drop_whitespace=False,
        )
        lines.extend(wrapped if wrapped else [""])
    return "\n".join(lines)

# ---------------------------
# PDF-safe wrapping functions
# ---------------------------
def wrap_path_for_pdf(value, max_chunk=34):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    parts = []
    token = ""
    separators = {"\\", "/", "_", "-", "=", "."}

    for ch in text:
        token += ch
        if ch in separators:
            parts.append(token)
            token = ""
    if token:
        parts.append(token)

    lines = []
    current = ""

    for part in parts:
        if len(current) + len(part) <= max_chunk:
            current += part
        else:
            if current:
                lines.append(current)
            if len(part) <= max_chunk:
                current = part
            else:
                subparts = textwrap.wrap(part, width=max_chunk, break_long_words=True, break_on_hyphens=True)
                if subparts:
                    lines.extend(subparts[:-1])
                    current = subparts[-1]
                else:
                    current = part

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def wrap_general_text_for_pdf(value, max_len=42):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""
    words = text.split()
    lines = []
    current = ""

    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_len:
            current = candidate
        else:
            if current:
                lines.append(current)
            if len(word) > max_len:
                word_chunks = textwrap.wrap(word, width=max_len, break_long_words=True, break_on_hyphens=True)
                lines.extend(word_chunks[:-1])
                current = word_chunks[-1] if word_chunks else word
            else:
                current = word

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

# ============================================================
# 3) NOTEBOOK DISCOVERY + PARSING
# ============================================================
def find_relevant_notebooks():
    patterns = [
        "*ingest*.ipynb",
        "*ingestion*.ipynb",
        "*collect*.ipynb",
        "*extract*.ipynb",
        "*load*.ipynb",
        "*validate*.ipynb",
        "*validation*.ipynb",
        "*.ipynb",
    ]
    matches = []
    for pattern in patterns:
        matches.extend(PROJECT_ROOT.rglob(pattern))

    cleaned = []
    for p in sorted(set(matches)):
        p_str = safe_str(p).lower()
        if ".ipynb_checkpoints" in p_str:
            continue
        if "\\venv\\" in p_str or "/venv/" in p_str:
            continue
        if "\\.venv\\" in p_str or "/.venv/" in p_str:
            continue
        if "\\site-packages\\" in p_str or "/site-packages/" in p_str:
            continue
        cleaned.append(p)

    priority = []
    others = []
    for p in cleaned:
        name = p.name.lower()
        if any(k in name for k in ["ingest", "ingestion", "collect", "extract", "load", "validate", "validation"]):
            priority.append(p)
        else:
            others.append(p)

    ordered = sorted(set(priority), key=lambda x: safe_str(x).lower()) + [
        p for p in sorted(set(others), key=lambda x: safe_str(x).lower()) if p not in set(priority)
    ]
    return ordered

def normalize_nb_text(value):
    if isinstance(value, list):
        return "".join(str(x) for x in value)
    return str(value)

def extract_output_text(output):
    try:
        output_type = output.get("output_type", "")

        if output_type == "stream":
            return normalize_nb_text(output.get("text", ""))

        if output_type in ["execute_result", "display_data"]:
            data = output.get("data", {})
            if "text/plain" in data:
                return normalize_nb_text(data["text/plain"])
            if "text/html" in data:
                return normalize_nb_text(data["text/html"])
            return "[Non-text output omitted]"

        if output_type == "error":
            traceback = output.get("traceback", [])
            if traceback:
                return "\n".join(traceback)
            ename = output.get("ename", "Error")
            evalue = output.get("evalue", "")
            return f"{ename}: {evalue}"

        return "[Unsupported output type]"
    except Exception as e:
        return f"Could not parse output: {e}"

def read_notebook_details(nb_path, max_code_cells=3, max_source_chars=1800, max_output_chars=1200):
    result = {
        "name": nb_path.name,
        "relative_path": rel_path(nb_path),
        "cells": [],
        "error": None,
    }

    try:
        with open(nb_path, "r", encoding="utf-8") as f:
            nb = json.load(f)

        code_cells = [cell for cell in nb.get("cells", []) if cell.get("cell_type") == "code"]

        for idx, cell in enumerate(code_cells[:max_code_cells], start=1):
            source = normalize_nb_text(cell.get("source", ""))
            source = source.strip()[:max_source_chars] if source else "No code found."

            outputs = cell.get("outputs", [])
            output_texts = []
            for out in outputs[:3]:
                output_texts.append(extract_output_text(out))

            output_combined = "\n\n".join(x for x in output_texts if x).strip()
            if not output_combined:
                output_combined = "No saved output found in notebook."
            output_combined = output_combined[:max_output_chars]

            result["cells"].append({
                "cell_no": idx,
                "source": source,
                "output": output_combined,
            })

        if not result["cells"]:
            result["cells"].append({
                "cell_no": 1,
                "source": "No code cells found in notebook.",
                "output": "No output available.",
            })

    except Exception as e:
        result["error"] = safe_str(e)
        result["cells"].append({
            "cell_no": 1,
            "source": f"Could not parse notebook: {e}",
            "output": "No output available.",
        })

    return result

# ============================================================
# 4) STYLES
# ============================================================
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    name="CustomTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    spaceAfter=14,
)

meta_style = ParagraphStyle(
    name="MetaStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=13,
    spaceAfter=5,
)

heading_style = ParagraphStyle(
    name="HeadingStyle",
    parent=styles["Heading2"],
    alignment=TA_LEFT,
    fontSize=12,
    leading=15,
    spaceAfter=8,
)

sub_heading_style = ParagraphStyle(
    name="SubHeadingStyle",
    parent=styles["Heading3"],
    alignment=TA_LEFT,
    fontSize=10.5,
    leading=13,
    spaceAfter=6,
)

body_style = ParagraphStyle(
    name="BodyStyle",
    parent=styles["BodyText"],
    alignment=TA_JUSTIFY,
    fontSize=10.0,
    leading=14,
    spaceAfter=8,
)

bullet_style = ParagraphStyle(
    name="BulletStyle",
    parent=styles["BodyText"],
    alignment=TA_LEFT,
    fontSize=10.0,
    leading=14,
    leftIndent=14,
    firstLineIndent=-8,
    spaceAfter=4,
)

code_style = ParagraphStyle(
    name="CodeStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=7.0,
    leading=8.4,
)

table_header_style = ParagraphStyle(
    name="TableHeaderStyle",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=8.1,
    leading=9.3,
    alignment=TA_LEFT,
)

table_cell_style = ParagraphStyle(
    name="TableCellStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=7.2,
    leading=8.6,
    alignment=TA_LEFT,
)

def to_para(value, style, kind="general"):
    if kind == "path":
        return Paragraph(wrap_path_for_pdf(value), style)
    return Paragraph(wrap_general_text_for_pdf(value), style)

def make_wrapped_table(data, col_widths=None, header_bg="#D9EAD3", path_cols=None, file_cols=None):
    path_cols = path_cols or []
    file_cols = file_cols or []

    converted = []
    for r, row in enumerate(data):
        row_cells = []
        for c, cell in enumerate(row):
            style = table_header_style if r == 0 else table_cell_style
            if r == 0:
                row_cells.append(Paragraph(escape(str(cell)), style))
            else:
                if c in path_cols or c in file_cols:
                    row_cells.append(to_para(cell, style, kind="path"))
                else:
                    row_cells.append(to_para(cell, style, kind="general"))
        converted.append(row_cells)

    table = Table(converted, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(header_bg)),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ]))
    return table

# ============================================================
# 5) COLLECT PROJECT EVIDENCE
# ============================================================
retailrocket_csvs = collect_files(
    RAW_ROOT,
    [
        "**/events.csv",
        "**/category_tree.csv",
        "**/item_properties_part1.csv",
        "**/item_properties_part2.csv",
    ],
)

dummyjson_raw = collect_files(
    RAW_ROOT,
    [
        "**/products_raw.json",
        "**/categories_raw.json",
    ],
)

bronze_files = collect_files(
    BRONZE_ROOT,
    [
        "**/*.parquet",
    ],
)

logs_found = find_log_files()

latest_validation_txt = latest_file(PROJECT_ROOT, ["**/run_validation.txt"])
latest_json_report = latest_file(REPORTS_DIR, ["**/*report*.json"])
latest_pdf_report = latest_file(REPORTS_DIR, ["**/*report*.pdf"])
latest_fix_log = latest_file(REPORTS_DIR, ["**/*fix*.csv"])
latest_summary_csv = latest_file(REPORTS_DIR, ["**/*summary*.csv"])
latest_issues_csv = latest_file(REPORTS_DIR, ["**/*issues*.csv"])

notebooks_found = find_relevant_notebooks()
notebook_details = [read_notebook_details(nb) for nb in notebooks_found[:6]]

print("\nDiscovered files:")
for name, path in [
    ("events_csv", retailrocket_csvs[0] if len(retailrocket_csvs) > 0 else None),
    ("category_tree_csv", retailrocket_csvs[1] if len(retailrocket_csvs) > 1 else None),
    ("item_properties_part1_csv", retailrocket_csvs[2] if len(retailrocket_csvs) > 2 else None),
    ("item_properties_part2_csv", retailrocket_csvs[3] if len(retailrocket_csvs) > 3 else None),
    ("products_raw_json", dummyjson_raw[0] if len(dummyjson_raw) > 0 else None),
    ("categories_raw_json", dummyjson_raw[1] if len(dummyjson_raw) > 1 else None),
    ("products_parquet", bronze_files[0] if len(bronze_files) > 0 else None),
    ("categories_parquet", bronze_files[1] if len(bronze_files) > 1 else None),
]:
    print(f"- {name}: {path if path else 'Not found'}")

print("\nRelevant notebooks found:")
if notebooks_found:
    for nb in notebooks_found[:10]:
        print(f"- {rel_path(nb)}")
else:
    print("- No notebooks found")

validation_preview_full = read_text_preview(latest_validation_txt, max_lines=60)
initial_status = "Unknown"
final_status = "Unknown"
for line in validation_preview_full.splitlines():
    if "Initial status:" in line:
        initial_status = line.split("Initial status:")[-1].strip()
    if "Final status:" in line:
        final_status = line.split("Final status:")[-1].strip()

print(f"\nInitial status: {initial_status}")
print(f"Final status: {final_status}")
print(f"Initial issues file: {latest_issues_csv if latest_issues_csv else 'Not found'}")
print(f"Initial summary file: {latest_summary_csv if latest_summary_csv else 'Not found'}")
print(f"Fix log file: {latest_fix_log if latest_fix_log else 'Not found'}")
print(f"JSON report file: {latest_json_report if latest_json_report else 'Not found'}")
print(f"PDF report file: {latest_pdf_report if latest_pdf_report else 'Not found'}")

source_summary_rows = []

for path in retailrocket_csvs:
    meta = file_info(path)
    csv_meta = summarize_csv(path)
    source_summary_rows.append([
        "RetailRocket CSV",
        path.name,
        rel_path(path),
        ", ".join(csv_meta.get("columns", [])[:8]) if csv_meta.get("columns") else (csv_meta.get("error") or "N/A"),
        meta["modified"] if meta else "N/A",
    ])

for path in dummyjson_raw:
    meta = file_info(path)
    js = summarize_json(path)
    attributes = js.get("sample_keys") or js.get("top_keys") or []
    source_summary_rows.append([
        "DummyJSON Raw JSON",
        path.name,
        rel_path(path),
        ", ".join(attributes[:8]) if attributes else (js.get("error") or "N/A"),
        meta["modified"] if meta else "N/A",
    ])

for path in bronze_files:
    meta = file_info(path)
    pq = summarize_parquet(path)
    source_summary_rows.append([
        "Bronze Parquet",
        path.name,
        rel_path(path),
        ", ".join(pq.get("columns", [])[:8]) if pq.get("columns") else (pq.get("error") or "N/A"),
        meta["modified"] if meta else "N/A",
    ])

if not source_summary_rows:
    source_summary_rows = [["No sources found", "-", "-", "-", "-"]]

notebook_rows = []
if notebooks_found:
    for nb in notebooks_found[:10]:
        meta = file_info(nb)
        notebook_rows.append([
            nb.name,
            rel_path(nb),
            meta["modified"] if meta else "N/A",
            f"{meta['size_kb']} KB" if meta else "N/A",
        ])
else:
    notebook_rows = [["No notebook found", "-", "-", "-"]]

log_rows = []
preferred_logs = [
    latest_validation_txt,
    latest_json_report,
    latest_pdf_report,
    latest_fix_log,
    latest_summary_csv,
    latest_issues_csv,
]
for lf in preferred_logs:
    if lf and lf.exists():
        meta = file_info(lf)
        log_rows.append([
            lf.name,
            rel_path(lf),
            meta["modified"],
            f"{meta['size_kb']} KB",
        ])

if not log_rows:
    for lf in logs_found[:10]:
        meta = file_info(lf)
        log_rows.append([
            lf.name,
            rel_path(lf),
            meta["modified"],
            f"{meta['size_kb']} KB",
        ])

if not log_rows:
    log_rows = [["No logs found", "-", "-", "-"]]

raw_tree = build_tree_text(RAW_ROOT, max_depth=5, max_items=200)
bronze_tree = build_tree_text(BRONZE_ROOT, max_depth=5, max_items=200)
validation_preview = read_text_preview(latest_validation_txt, max_lines=35) if latest_validation_txt else "run_validation.txt not found."
json_report_preview = read_text_preview(latest_json_report, max_lines=35) if latest_json_report else "JSON report not found."

# ============================================================
# 6) TABLES
# ============================================================
team_data = [
    ["Team Member Name", "Team Member ID"],
    ["BANSHIDHAR RATH", "2025AE05346"],
    ["JITENDRA KUMAR TIWARI", "2025AE05518"],
    ["KATBA ANKIT CHIMANBHAI", "2025AE05229"],
    ["NAVEEN SURATHU", "2025AE05492"],
]

source_table_data = [["Source Type", "File Name", "Relative Path", "Key Attributes / Columns", "Last Modified"]]
source_table_data.extend(source_summary_rows)

notebook_table_data = [["Notebook Name", "Relative Path", "Last Modified", "Size"]]
notebook_table_data.extend(notebook_rows)

log_table_data = [["Log / Report File", "Relative Path", "Last Modified", "Size"]]
log_table_data.extend(log_rows)

team_table = make_wrapped_table(
    team_data,
    col_widths=[4.0 * inch, 2.0 * inch]
)

source_table = make_wrapped_table(
    source_table_data,
    col_widths=[1.10 * inch, 1.35 * inch, 2.10 * inch, 1.65 * inch, 1.05 * inch],
    path_cols=[2],
    file_cols=[1]
)

notebook_table = make_wrapped_table(
    notebook_table_data,
    col_widths=[1.75 * inch, 3.35 * inch, 0.95 * inch, 0.55 * inch],
    path_cols=[1],
    file_cols=[0]
)

log_table = make_wrapped_table(
    log_table_data,
    col_widths=[1.85 * inch, 3.10 * inch, 0.95 * inch, 0.55 * inch],
    path_cols=[1],
    file_cols=[0]
)

# ============================================================
# 7) BUILD STORY
# ============================================================
story = []

story.append(Paragraph("02 Data Collection and Ingestion", title_style))
story.append(Paragraph("<b>Course Name:</b> Data Management for Machine Learning", meta_style))
story.append(Paragraph("<b>Assignment Title:</b> End-to-End Data Management Pipeline for a Recommendation System", meta_style))
story.append(Paragraph("<b>Assignment:</b> Group 51 - Data management for Machine Learning Group 51", meta_style))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>Team Members</b>", heading_style))
story.append(team_table)
story.append(Spacer(1, 14))

story.append(Paragraph("1. Objective of Data Collection and Ingestion", heading_style))
story.append(Paragraph(
    "This section documents how the recommendation pipeline ingests data from multiple source types, stores collected data in structured raw and bronze layers, and preserves logs and reports for monitoring and auditability. The evidence below is automatically collected from the current project folder so that the PDF reflects the actual implementation state.",
    body_style
))

story.append(Paragraph("2. Ingestion Summary", heading_style))
story.append(Paragraph(
    "The project collects at least two categories of source data: behavioral and catalog CSV files, and external API-style JSON product data. The notebook scans the project folders and summarizes the discovered ingestion assets below.",
    body_style
))
story.append(source_table)
story.append(Spacer(1, 12))

story.append(Paragraph("3. Data Sources and Attributes", heading_style))
story.append(Paragraph("• RetailRocket CSV files capture user interactions and item metadata such as events, categories, timestamps, and item properties.", bullet_style))
story.append(Paragraph("• DummyJSON raw JSON files provide product and category information used to enrich item-level context.", bullet_style))
story.append(Paragraph("• Bronze parquet datasets represent ingested and standardized downstream storage for curated use.", bullet_style))
story.append(Paragraph("• Together, these sources support recommendation use cases such as interaction modeling, item enrichment, and downstream validation.", bullet_style))
story.append(Spacer(1, 10))

story.append(Paragraph("4. Ingestion and Validation Notebooks Used", heading_style))
story.append(Paragraph(
    "The following notebooks were identified from the project as relevant to ingestion or validation activities. Their notebook names, locations, saved code cells, and captured outputs are shown below.",
    body_style
))
story.append(notebook_table)
story.append(Spacer(1, 12))

for nb in notebook_details:
    story.append(Paragraph(f"Notebook: {escape(nb['name'])}", sub_heading_style))
    story.append(Paragraph(f"<b>Path:</b> {escape(nb['relative_path'])}", meta_style))

    if nb.get("error"):
        story.append(Paragraph(f"<b>Parse Note:</b> {escape(nb['error'])}", meta_style))

    for cell in nb["cells"]:
        story.append(Paragraph(f"Code Cell {cell['cell_no']}", meta_style))
        story.append(Preformatted(wrap_block_text(cell["source"], width=95), code_style))
        story.append(Spacer(1, 4))

        story.append(Paragraph(f"Output Cell {cell['cell_no']}", meta_style))
        story.append(Preformatted(wrap_block_text(cell["output"], width=95), code_style))
        story.append(Spacer(1, 8))

story.append(Paragraph("5. Logging and Audit Trail Evidence", heading_style))
story.append(Paragraph(
    f"The latest execution evidence indicates an initial status of <b>{escape(initial_status)}</b> and a final status of <b>{escape(final_status)}</b>. The following artifacts provide supporting evidence for ingestion, validation, remediation, and reporting.",
    body_style
))
story.append(log_table)
story.append(Spacer(1, 12))

story.append(Paragraph("6. Raw and Bronze Storage Structure", heading_style))
story.append(Paragraph(
    "A structured folder layout is used to store ingested data. Raw files are organized under source-specific directories, while bronze data contains normalized parquet outputs for downstream processing.",
    body_style
))

story.append(Paragraph("<b>Raw Data Folder Tree</b>", meta_style))
story.append(Preformatted(wrap_block_text(raw_tree, width=90), code_style))
story.append(Spacer(1, 8))

story.append(Paragraph("<b>Bronze Data Folder Tree</b>", meta_style))
story.append(Preformatted(wrap_block_text(bronze_tree, width=90), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("7. Validation / Execution Log Preview", heading_style))
story.append(Paragraph(
    "The preview below captures the beginning of the latest validation or execution text file discovered in the project. This provides traceable evidence of discovered source files, execution statuses, and generated outputs.",
    body_style
))
story.append(Preformatted(wrap_block_text(validation_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("8. JSON Report Preview", heading_style))
story.append(Paragraph(
    "If a JSON report exists, the preview below captures the first lines for quick inspection of generated reporting metadata.",
    body_style
))
story.append(Preformatted(wrap_block_text(json_report_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("9. Conclusion", heading_style))
story.append(Paragraph(
    "Based on the discovered project structure, the pipeline demonstrates multi-source data ingestion, structured raw and bronze storage, and generation of logs and reports that support monitoring and auditability. This PDF was generated directly from the available project files to create submission-ready documentation for the Data Collection and Ingestion stage.",
    body_style
))

# ============================================================
# 8) BUILD PDF WITH FILE-LOCK FALLBACK
# ============================================================
def build_pdf(path):
    doc = SimpleDocTemplate(
        str(path),
        pagesize=A4,
        rightMargin=0.50 * inch,
        leftMargin=0.50 * inch,
        topMargin=0.55 * inch,
        bottomMargin=0.55 * inch,
    )
    doc.build(story)

try:
    build_pdf(OUTPUT_PATH)
    print(f"\nPDF created successfully: {OUTPUT_PATH}")
except PermissionError:
    alt_path = PROJECT_ROOT / f"02 Data Collection and Ingestion- DM4ML-Group51-{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    build_pdf(alt_path)
    print("\nOriginal output file is likely open or locked.")
    print(f"Saved alternate file instead: {alt_path}")


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
RAW_ROOT: C:\Users\barath\recomart-pipeline\data\raw
BRONZE_ROOT: C:\Users\barath\recomart-pipeline\data\bronze

Discovered files:
- events_csv: C:\Users\barath\recomart-pipeline\data\raw\retailrocket\load_date=2026-04-29\load_hour=11\category_tree.csv
- category_tree_csv: C:\Users\barath\recomart-pipeline\data\raw\retailrocket\load_date=2026-04-29\load_hour=11\events.csv
- item_properties_part1_csv: C:\Users\barath\recomart-pipeline\data\raw\retailrocket\load_date=2026-04-29\load_hour=11\item_properties_part1.csv
- item_properties_part2_csv: C:\Users\barath\recomart-pipeline\data\raw\retailrocket\load_date=2026-04-29\load_hour=11\item_properties_part2.csv
- products_raw_json: C:\Users\barath\recomart-pipeline\data\raw\dummyjson\load_date=2026-04-29\load_hour=11\categories_raw.json
- categories_raw_json: C:\Users\barath\recomart-pipeline\data\raw\dummyjson\load_date=2026-04-29\load_hour=11\products_raw.json
- products_parquet: C:\Users\ba